# Use case: Predict patient readmission risk using PySpark ML by performing feature engineering on patient health/admission data and training a Random Forest model.

## Flow: Dataset → Feature Engineering → Random Forest → Risk Probability → High/Medium/Low Risk → Gold Delta Table → Power BI.

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
import random
from datetime import datetime, timedelta
import builtins

spark = SparkSession.builder.getOrCreate()

# Number of records
num_records = 100000

# Generate patient data
data = []

for i in range(1, num_records + 1):

    age = random.randint(18, 85)

    gender = random.choice(["Male", "Female"])

    bmi = builtins.round(random.uniform(16, 40), 2)

    blood_pressure = random.randint(90, 180)

    hba1c = builtins.round(random.uniform(4.5, 10.5), 2)

    cholesterol = random.randint(120, 300)

    chronic_disease = random.choice([
        "Diabetes",
        "Hypertension",
        "Heart Disease",
        "None"
    ])

    previous_admissions = random.randint(0, 5)

    emergency_visit = random.choice([0, 1])

    medication_count = random.randint(0, 10)

    length_of_stay = random.randint(1, 15)

    # Create target variable
    risk_score = 0

    if age > 60:
        risk_score += 1

    if bmi > 30:
        risk_score += 1

    if blood_pressure > 140:
        risk_score += 1

    if hba1c > 7:
        risk_score += 1

    if previous_admissions >= 2:
        risk_score += 1

    if emergency_visit == 1:
        risk_score += 1

    if chronic_disease != "None":
        risk_score += 1

    readmission = 1 if risk_score >= 3 else 0

    admission_date = datetime.now() - timedelta(
        days=random.randint(1, 1000)
    )

    data.append((
        i,
        age,
        gender,
        bmi,
        blood_pressure,
        hba1c,
        cholesterol,
        chronic_disease,
        previous_admissions,
        emergency_visit,
        medication_count,
        length_of_stay,
        admission_date,
        readmission
    ))

columns = [
    "patient_id",
    "age",
    "gender",
    "bmi",
    "blood_pressure",
    "hba1c",
    "cholesterol",
    "chronic_disease",
    "previous_admissions",
    "emergency_visit",
    "medication_count",
    "length_of_stay",
    "admission_date",
    "readmission"
]

df = spark.createDataFrame(data, columns)

display(df)

### Data Quality Layer

In [0]:
print("Total records:", df.count())

print(
    "Duplicate patient IDs:",
    df.select("patient_id").distinct().count()
)

In [0]:
ml_df = df.dropDuplicates(["patient_id"])

### Feature Engineering

In [0]:
ml_df = ml_df.withColumn(
    "age_group",
    when(col("age") < 30, "Young")
    .when(col("age") < 50, "Adult")
    .when(col("age") < 65, "Middle_Aged")
    .otherwise("Senior")
)

In [0]:
ml_df = ml_df.withColumn(
    "bmi_category",
    when(col("bmi") < 18.5, "Underweight")
    .when(col("bmi") < 25, "Normal")
    .when(col("bmi") < 30, "Overweight")
    .otherwise("Obese")
)

In [0]:
ml_df = ml_df.withColumn(
    "high_bp_flag",
    when(col("blood_pressure") >= 140, 1).otherwise(0)
)

In [0]:
ml_df = ml_df.withColumn(
    "previous_admission_flag",
    when(col("previous_admissions") >= 2, 1).otherwise(0)
)

In [0]:
ml_df = ml_df.withColumn(
    "emergency_flag",
    when(col("emergency_visit") == 1, 1).otherwise(0)
)

In [0]:
ml_df = ml_df.withColumn(
    "high_medication_flag",
    when(col("medication_count") >= 5, 1).otherwise(0)
)

In [0]:
ml_df = ml_df.withColumn(
    "high_hba1c_flag",
    when(col("hba1c") >= 7, 1).otherwise(0)
)

### Create a Clinical Risk Score

In [0]:
ml_df = ml_df.withColumn(
    "clinical_risk_score",
    col("high_bp_flag") +
    col("high_hba1c_flag") +
    col("previous_admission_flag") +
    col("emergency_flag") +
    col("high_medication_flag") +
    when(col("age") >= 60, 1).otherwise(0) +
    when(col("bmi") >= 30, 1).otherwise(0) +
    when(col("chronic_disease") != "None", 1).otherwise(0)
)

In [0]:
ml_df = ml_df.withColumn(
    "clinical_risk_category",
    when(col("clinical_risk_score") >= 5, "High")
    .when(col("clinical_risk_score") >= 3, "Medium")
    .otherwise("Low")
)

In [0]:
from pyspark.ml.feature import StringIndexer

gender_indexer = StringIndexer(
    inputCol="gender",
    outputCol="gender_index",
    handleInvalid="keep"
)

disease_indexer = StringIndexer(
    inputCol="chronic_disease",
    outputCol="disease_index",
    handleInvalid="keep"
)

In [0]:
from pyspark.ml.feature import VectorAssembler

feature_columns = [
    "age",
    "bmi",
    "blood_pressure",
    "hba1c",
    "cholesterol",
    "previous_admissions",
    "emergency_visit",
    "medication_count",
    "length_of_stay",
    "clinical_risk_score",
    "gender_index",
    "disease_index"
]

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

In [0]:
train_df, test_df = ml_df.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Training records:", train_df.count())
print("Testing records:", test_df.count())

### Build the AI/ML Model

In [0]:
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    labelCol="readmission",
    featuresCol="features",
    numTrees=100,
    maxDepth=8,
    seed=42
)

In [0]:
from pyspark.ml import Pipeline

pipeline = Pipeline(
    stages=[
        gender_indexer,
        disease_indexer,
        assembler,
        rf
    ]
)

In [0]:
model = pipeline.fit(train_df)

In [0]:
predictions = model.transform(test_df)

In [0]:
display(
    predictions.select(
        "patient_id",
        "readmission",
        "prediction",
        "probability"
    )
)

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType

get_probability = udf(
    lambda v: float(v[1]),
    DoubleType()
)

predictions = predictions.withColumn(
    "readmission_probability",
    get_probability(col("probability"))
)

In [0]:
predictions = predictions.withColumn(
    "predicted_risk_category",
    when(
        col("readmission_probability") >= 0.70,
        "High Risk"
    )
    .when(
        col("readmission_probability") >= 0.40,
        "Medium Risk"
    )
    .otherwise("Low Risk")
)

In [0]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

auc_evaluator = BinaryClassificationEvaluator(
    labelCol="readmission",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = auc_evaluator.evaluate(predictions)

print("AUC:", auc)

In [0]:
from pyspark.sql.functions import avg

accuracy = predictions.select(
    avg(
        (col("prediction") == col("readmission"))
        .cast("double")
    ).alias("accuracy")
).collect()[0]["accuracy"]

print("Accuracy:", accuracy)

### Feature Importance

In [0]:
rf_model = model.stages[-1]

In [0]:
importance = rf_model.featureImportances

for feature, score in zip(feature_columns, importance):
    print(feature, ":", float(score))

In [0]:
feature_importance = spark.createDataFrame(
    [
        (feature, float(score))
        for feature, score in zip(
            feature_columns,
            importance
        )
    ],
    ["feature", "importance"]
)

display(
    feature_importance.orderBy(
        col("importance").desc()
    )
)

In [0]:
gold_df = predictions.select(
    "patient_id",
    "age",
    "gender",
    "bmi",
    "blood_pressure",
    "hba1c",
    "cholesterol",
    "chronic_disease",
    "previous_admissions",
    "emergency_visit",
    "medication_count",
    "length_of_stay",
    "clinical_risk_score",
    "clinical_risk_category",
    "readmission",
    "prediction",
    "readmission_probability",
    "predicted_risk_category"
)